# Notebook 6: Long-Term Memory — PROCEDURAL (Rules & Behavior)

### What it does (simplest version)
The agent's **instructions themselves** — its system prompt, its rules of behavior —
live in the store. And they can be **rewritten from feedback**. The agent literally
reprograms itself.

### The three long-term types, side by side
| Type | Remembers... | Changes... |
|---|---|---|
| Semantic (NB4) | facts & preferences | what the agent **knows** |
| Episodic (NB5) | past experiences | what the agent **does** (copies what worked) |
| Procedural (NB6) | rules & instructions | what the agent **is** (its behavior itself) |

### Human analogy
An employee who, after feedback from their manager, **rewrites their own job instructions**:
*"From now on: shorter emails, always attach the report."*
Next week — new task, new person asking — the improved behavior is already there.

### Why store the prompt instead of hard-coding it?
A hard-coded system prompt means: edit code → redeploy → everyone gets the change at once.
A stored prompt means: feedback in → better instructions out → **no redeploy**, and it can
even differ per user/team.

### Where to use it
Agents that adapt tone/format to a user ("shorter answers", "always show code"),
self-improving assistants, team-specific behavior rules.

### What it connects with
This completes the map: checkpointer (short-term) + store with semantic, episodic,
and procedural memory (long-term). The final cell recaps the whole picture.

## Step 1 — Setup: put the system prompt IN THE STORE

Notice: the prompt is **data in the store**, not a string in the code.
Namespace: `("agents", "py-tutor", "prompts")` — the rules belong to this agent.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.store.memory import InMemoryStore

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
store = InMemoryStore()

NS = ("agents", "py-tutor", "prompts")

# The agent's behavior rules start simple...
store.put(NS, key="system_prompt",
          value={"text": "You are a Python tutor. Explain concepts clearly."})

print("Stored instructions:", store.get(NS, "system_prompt").value["text"])

## Step 2 — Ask a question (the agent reads its rules from the store first)

The `ask()` helper shows the pattern in its purest form:
**read rules from store → build system message → call LLM.** Every single time.

In [ ]:
def ask(question: str) -> str:
    current_rules = store.get(NS, "system_prompt").value["text"]   # ★ read rules
    response = llm.invoke([
        SystemMessage(content=current_rules),
        HumanMessage(content=question),
    ])
    return response.content

print("BEFORE feedback:")
print(ask("What is a Python decorator?"))

## Step 3 — The feedback loop: rewrite the stored rules

A user says: *"too long — short code example every time, max 3 sentences."*

Instead of us editing code, we ask the LLM to **rewrite the stored instructions**
with the feedback baked in, and `store.put` them back. From now on, **every** future
conversation gets the improved behavior.

In [ ]:
def update_from_feedback(feedback: str):
    """Rewrite the stored instructions to incorporate feedback."""
    current_rules = store.get(NS, "system_prompt").value["text"]
    new_rules = llm.invoke(
        "Here are an assistant's current instructions:
"
        f"{current_rules}

"
        f"A user gave this feedback: {feedback}

"
        "Rewrite the instructions to incorporate the feedback. "
        "Keep them short. Return ONLY the new instructions."
    )
    store.put(NS, key="system_prompt", value={"text": new_rules.content})  # ★ write rules

update_from_feedback(
    "Your answers are too long. Use a short code example every time, "
    "and explain in max 3 sentences."
)
print("Rules updated. New stored instructions:
")
print(store.get(NS, "system_prompt").value["text"])

## Step 4 — Same question, new behavior

We changed **no code** since Step 2 — only the data in the store.
Compare the answer with the "BEFORE" one.

In [ ]:
print("AFTER feedback (same question):")
print(ask("What is a Python decorator?"))

## Try it yourself

1. Give more feedback: *"Also, always end with one practice exercise."* — run
   `update_from_feedback(...)` again and watch the rules evolve a second time.
2. Discussion: semantic vs procedural — if the user says *"I prefer short answers"*,
   is that a **fact about the user** (semantic) or a **rule for the agent** (procedural)?
   There's a real design judgment here; both can work.

## The complete picture (final recap)

```
LANGCHAIN MEMORY
├── SHORT-TERM (current thread)         → Agent State + Checkpointer
│     ├── full history        (NB1)  — remembers everything in this thread
│     ├── window / trim       (NB2)  — cheap, forgets old turns
│     └── summary             (NB3)  — compresses old turns
└── LONG-TERM (across threads)          → Store
      ├── SEMANTIC            (NB4)  — facts & preferences  (what it KNOWS)
      ├── EPISODIC            (NB5)  — past experiences     (what it DID)
      └── PROCEDURAL          (NB6)  — rules & behavior     (what it IS)
```

### Homework: the study-buddy bot
Combine all layers in one agent:
1. Checkpointer for short-term memory (`thread_id` per study session).
2. Summarize after 8 messages (NB3 pattern).
3. Semantic memory: store the student's name, exam date, weak topics, keyed by `user_id`.
4. Procedural memory: after feedback like *"explain more simply"*, update the stored rules.
5. Test: start a **new thread** — the bot should greet the student by name,
   recall the weak topics, and keep its improved teaching style.

Build that, and you've built every kind of memory in the 2026 LangChain stack.